# RunPod End-to-End Notebook (System Python, No venv)

This notebook is designed for RunPod Jupyter and runs the `luminance-based-diffusion` pipeline using system Python.

It includes clone/pull from:
- `https://github.com/Nasapan23/luminance-based-diffusion`

Notes:
- No virtual environment is created.
- It assumes CUDA PyTorch is already present in your RunPod image.
- Heavy stages are controlled by flags in the next cell.


In [ ]:
import os
from pathlib import Path

# Required repo settings
REPO_URL = "https://github.com/Nasapan23/luminance-based-diffusion"
REPO_DIR = "/workspace/luminance-based-diffusion"
BRANCH = "main"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()

# Optional Hugging Face token: set in RunPod env or paste here.
# For SDXL base, your HF account should have accepted model license terms.
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Pipeline toggles
RUN_REAL20K = 0          # Optional manual run of scripts/run_real_20k.sh
RUN_SDXL_TRAIN = 1       # Runs full SDXL grayscale training (auto-prepares data if missing)
RUN_VAZE_PREP = 0        # 1 = run amphora local prep from data/vaze (only if needed)
RUN_INFER_DRY = 0        # 1 = run ComfyUI dry-run inference commands
RUN_INFER_FULL = 0       # 1 = run live ComfyUI inference commands (requires API server)

# Amphora prep control (0 = all valid images, else cap)
VAZE_MAX_IMAGES = 0

env_map = {
    "REPO_URL": REPO_URL,
    "REPO_DIR": REPO_DIR,
    "BRANCH": BRANCH,
    "GITHUB_TOKEN": GITHUB_TOKEN,
    "HF_TOKEN": HF_TOKEN,
    "RUN_REAL20K": str(RUN_REAL20K),
    "RUN_SDXL_TRAIN": str(RUN_SDXL_TRAIN),
    "RUN_VAZE_PREP": str(RUN_VAZE_PREP),
    "RUN_INFER_DRY": str(RUN_INFER_DRY),
    "RUN_INFER_FULL": str(RUN_INFER_FULL),
    "VAZE_MAX_IMAGES": str(VAZE_MAX_IMAGES),
}

env_file = Path("/workspace/lbd_runpod.env")
with env_file.open("w", encoding="utf-8") as f:
    for k, v in env_map.items():
        os.environ[k] = v
        safe = v.replace('"', '\\"')
        f.write(f'export {k}="{safe}"\n')

print(f"Wrote env file: {env_file}")
for k in ["REPO_URL", "REPO_DIR", "BRANCH", "RUN_SDXL_TRAIN", "RUN_VAZE_PREP"]:
    print(f"{k}={os.environ[k]}")


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env

mkdir -p /workspace

AUTH_REPO_URL="$REPO_URL"
if [ -n "${GITHUB_TOKEN:-}" ]; then
  AUTH_REPO_URL="https://${GITHUB_TOKEN}@${REPO_URL#https://}"
fi

if [ -d "$REPO_DIR/.git" ]; then
  echo "Repo exists. Syncing..."
  git -C "$REPO_DIR" remote set-url origin "$AUTH_REPO_URL" || true
  git -C "$REPO_DIR" fetch origin --prune
  if git -C "$REPO_DIR" show-ref --verify --quiet "refs/heads/$BRANCH"; then
    git -C "$REPO_DIR" checkout "$BRANCH"
  else
    git -C "$REPO_DIR" checkout -B "$BRANCH" "origin/$BRANCH"
  fi
  git -C "$REPO_DIR" pull --ff-only origin "$BRANCH"
else
  echo "Cloning repo..."
  git clone --branch "$BRANCH" "$AUTH_REPO_URL" "$REPO_DIR"
fi

# Explicit pull from provided URL (as requested)
git -C "$REPO_DIR" pull --ff-only "$AUTH_REPO_URL" "$BRANCH"
if [ -n "${GITHUB_TOKEN:-}" ]; then
  git -C "$REPO_DIR" remote set-url origin "$REPO_URL" || true
fi

echo "Current commit:"
git -C "$REPO_DIR" rev-parse --short HEAD


In [ ]:
%%bash
set -euo pipefail

if command -v apt-get >/dev/null 2>&1; then
  export DEBIAN_FRONTEND=noninteractive
  apt-get update -y
  apt-get install -y git curl wget build-essential python3-dev libgl1 libglib2.0-0
fi

python -m pip install --upgrade pip setuptools wheel
python -m pip install --upgrade "huggingface_hub[cli]"


In [ ]:
import subprocess
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))

from torchvision.ops import nms
print("torchvision nms op: OK")

subprocess.run(["nvidia-smi"], check=False)


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env
cd "$REPO_DIR"

check_torch_stack() {
python - <<'PY'
import sys
import torch
print('torch', torch.__version__)
try:
    import torchvision
    print('torchvision', torchvision.__version__)
    from torchvision.ops import nms
    print('torchvision nms op: OK')
except Exception as exc:
    print('TORCHVISION_CHECK_FAILED:', repr(exc))
    sys.exit(2)
PY
}

# System install only (no venv)
python -m pip install -e ".[dev,train]"

# Keep train deps pinned as expected by repo scripts
python -m pip install "transformers>=4.41,<5"

if ! check_torch_stack; then
  echo "Repairing torch/torchvision/torchaudio stack (CUDA wheels)..."
  if ! python -m pip install --upgrade --force-reinstall --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio; then
    python -m pip install --upgrade --force-reinstall --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
  fi
  check_torch_stack
fi

# Intentionally skipping xformers install for stability in system Python images.


In [ ]:
import os
import subprocess

token = os.environ.get("HF_TOKEN", "").strip()
if token:
    print("Logging into Hugging Face CLI from HF_TOKEN...")
    subprocess.run(["huggingface-cli", "login", "--token", token], check=True)
else:
    print("HF_TOKEN is empty. If model download fails, set HF_TOKEN and rerun this cell.")


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env
cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src${PYTHONPATH:+:$PYTHONPATH}"

bash scripts/setup_diffusers_examples.sh

python -m lbd.cli train sdxl --config configs/train_sdxl.yaml --dry-run


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env
cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src${PYTHONPATH:+:$PYTHONPATH}"

if [ "$RUN_REAL20K" = "1" ]; then
  bash scripts/run_real_20k.sh
else
  echo "Skipping RUN_REAL20K (set RUN_REAL20K=1 in config cell to enable)."
fi


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env
cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src${PYTHONPATH:+:$PYTHONPATH}"

if [ "$RUN_SDXL_TRAIN" = "1" ]; then
  if [ ! -d "data/base20k/gray/train" ] || [ "$(find data/base20k/gray/train -type f \( -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.png' -o -iname '*.webp' \) | head -n 1 | wc -l)" -eq 0 ]; then
    echo "Dataset missing: auto-running scripts/run_real_20k.sh before SDXL training..."
    bash scripts/run_real_20k.sh
  fi
  python -m lbd.cli train sdxl --config configs/train_sdxl.yaml
else
  echo "Skipping RUN_SDXL_TRAIN (set RUN_SDXL_TRAIN=1 in config cell to enable)."
fi


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env
cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src${PYTHONPATH:+:$PYTHONPATH}"

if [ "$RUN_VAZE_PREP" = "1" ]; then
  echo "Expect your amphora images under: $REPO_DIR/data/vaze"
  VAZE_MAX_IMAGES="$VAZE_MAX_IMAGES" bash scripts/run_prepare_vaze_bw.sh
else
  echo "Skipping RUN_VAZE_PREP (set RUN_VAZE_PREP=1 in config cell to enable)."
fi


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env
cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src${PYTHONPATH:+:$PYTHONPATH}"

if [ "$RUN_INFER_DRY" = "1" ]; then
  python -m lbd.cli infer graygen --config configs/infer_graygen_comfyui.yaml --dry-run
  python -m lbd.cli infer recolor --config configs/infer_recolor_comfyui.yaml --dry-run
  python -m lbd.cli infer refine --config configs/infer_refine_comfyui.yaml --dry-run
else
  echo "Skipping RUN_INFER_DRY (set RUN_INFER_DRY=1 in config cell to enable)."
fi

if [ "$RUN_INFER_FULL" = "1" ]; then
  python -m lbd.cli infer graygen --config configs/infer_graygen_comfyui.yaml
  python -m lbd.cli infer recolor --config configs/infer_recolor_comfyui.yaml
  python -m lbd.cli infer refine --config configs/infer_refine_comfyui.yaml
else
  echo "Skipping RUN_INFER_FULL (set RUN_INFER_FULL=1 in config cell to enable)."
fi


In [ ]:
%%bash
set -euo pipefail
source /workspace/lbd_runpod.env
cd "$REPO_DIR"

echo "Recent runs:"
ls -lah runs || true

echo "GPU snapshot:"
nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv,noheader || true
